In [ ]:
import os
from copy import deepcopy
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
import torchvision
from torchvision.datasets import STL10
from torchvision import transforms
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

In [ ]:
DATASET_PATH = "..//data"
CHECKPOINT_PATH = "./saved_models/tutorial17"
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
%load_ext tensorboard
log_dir = "./saved_models/tutorial17/"
os.makedirs(log_dir, exist_ok=True)
%tensorboard --logdir {log_dir}

In [ ]:
#返回两个经数据增强后的图像
class ContrastiveTransformations(object):

    def __init__(self, base_transforms, n_views=2):
        self.base_transforms = base_transforms
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transforms(x) for i in range(self.n_views)]
#augmentation的五种方式
contrast_transforms = transforms.Compose([transforms.RandomHorizontalFlip(0.5),
                                          
                                          transforms.RandomResizedCrop(size=96),
                                            #颜色调整概率0.8
                                          transforms.RandomApply([
                                              transforms.ColorJitter(
                                                    #限制亮度在[0.5-1.5]
                                                  brightness=0.5,
                                                  #对比度
                                                    contrast=0.5,
                                                    #饱和度
                                                    saturation=0.5,
                                                    #色相偏移
                                                    hue=0.1)
                                          ], p=0.8),
                                          #转为灰度图像
                                          transforms.RandomGrayscale(p=0.2),
                                          #高斯卷积核模糊处理
                                          transforms.RandomApply(
                                            [ transforms.GaussianBlur(
                                                    kernel_size=9,
                                                    sigma=(0.1, 2.0)
                                                ) ],p=0.5),
                                          #totensor会将数据从（0，255）转到（0，1）
                                          transforms.ToTensor(),
                                          transforms.Normalize((0.5,), (0.5,))
                                         
                                         ])

In [ ]:
unlabeled_data = STL10(root=DATASET_PATH, split='unlabeled', download=True,
                       transform=ContrastiveTransformations(contrast_transforms, n_views=2))
train_data_contrast = STL10(root=DATASET_PATH, split='train', download=True,
                            transform=ContrastiveTransformations(contrast_transforms, n_views=2))

In [ ]:
class MoCo(pl.LightningModule):
    def __init__(self,
        hidden_dim,
        lr,
        temperature,
        weight_decay,
        max_epochs=500,
        queue_size=4096,
        momentum=0.999
    ):
        super().__init__()

        self.save_hyperparameters()

        assert temperature > 0
        assert 0 <= momentum < 1
        assert queue_size > 0

        # query encoder：通过反向传播更新
        self.encoder_q = self._build_encoder(hidden_dim)
        # key encoder：通过动量更新
        self.encoder_k = self._build_encoder(hidden_dim)

        # 两个编码器初始参数完全相同
        self.encoder_k.load_state_dict(self.encoder_q.state_dict())

        # key encoder 不参与反向传播
        for param in self.encoder_k.parameters():
            param.requires_grad = False

        # 负样本队列，形状为 [hidden_dim, queue_size]
        queue = torch.randn(hidden_dim, queue_size)
        queue = F.normalize(queue, dim=0)

        # register_buffer：
        # 1. 不参与梯度更新
        # 2. 会跟随模型移动到 GPU
        # 3. 会被保存到 checkpoint
        self.register_buffer("queue", queue)

        # 当前应该写入队列的位置
        self.register_buffer(
            "queue_ptr",
            torch.zeros(1, dtype=torch.long)
        )
    #对momentum_encoder进行动量更新
    @torch.no_grad()
    def _momentum_update_key_encoder(self):
        """
        使用 query encoder 的参数，
        对 key encoder 进行动量更新。
        """
        momentum = self.hparams.momentum

        for param_q, param_k in zip(
            self.encoder_q.parameters(),
            self.encoder_k.parameters()
        ):
            #相当于k*momentum
            param_k.data.mul_(momentum)
            param_k.data.add_(
                param_q.data,
                alpha=1.0 - momentum
            )
    #将新的key放入queue
    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        """
        把当前batch的key特征写入负样本队列。

        keys形状：[batch_size, hidden_dim]
        queue形状：[hidden_dim, queue_size]
        """
        keys = keys.detach()

        batch_size = keys.shape[0]
        queue_size = self.queue.shape[1]
        ptr = int(self.queue_ptr.item())

        end_ptr = ptr + batch_size

        if end_ptr <= queue_size:
            # 没有超过队列末尾，直接写入
            self.queue[:, ptr:end_ptr] = keys.T
        else:
            # 超过队列末尾，需要从队列开头继续写
            first_length = queue_size - ptr
            second_length = end_ptr - queue_size

            self.queue[:, ptr:] = keys[:first_length].T
            self.queue[:, :second_length] = keys[first_length:].T

        # 更新队列指针
        self.queue_ptr[0] = end_ptr % queue_size
    #创建一个resnet18网络和mlp
    def _build_encoder(self, hidden_dim):
        """创建一个 ResNet18 编码器和投影头。"""

        encoder = torchvision.models.resnet18(
            weights=None,
            num_classes=4 * hidden_dim
        )

        # STL10 是 96×96，使用较小的卷积头
        encoder.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False
        )

        encoder.maxpool = nn.Identity()

        # projection head
        encoder.fc = nn.Sequential(
            encoder.fc,
            nn.ReLU(inplace=True),
            nn.Linear(4 * hidden_dim, hidden_dim)
        )

        return encoder
    #opt和schduler
    def configure_optimizers(self):
        trainable_parameters = [
            param
            for param in self.encoder_q.parameters()
            if param.requires_grad]

        optimizer = optim.AdamW(
            trainable_parameters,
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )

        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.max_epochs,
            eta_min=self.hparams.lr / 50
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1}}
    #计算infonce
    def moco_loss(self, batch, mode="train"):
        images, _ = batch
        im_q, im_k = images

        # q形状：[B, hidden_dim]
        q = self.encoder_q(im_q)
        q = F.normalize(q, dim=1)

        with torch.no_grad():

            # 只有训练阶段才更新key encoder
            if mode == "train":
                self._momentum_update_key_encoder()
            # k形状：[B, hidden_dim]
            k = self.encoder_k(im_k)
            k = F.normalize(k, dim=1)

        # positive_logits形状：[B, 1]
        positive_logits = torch.sum(
            q * k,dim=1,keepdim=True)

        # negative_logits形状：[B, queue_size]
        negative_queue = self.queue.clone().detach()
        #进行向量相乘
        negative_logits = torch.matmul(
            q,
            negative_queue)

        # logits形状：[B, 1 + queue_size]
        logits = torch.cat(
            [positive_logits, negative_logits],
            dim=1)

        logits = logits / self.hparams.temperature

        # 正样本永远位于第0列
        labels = torch.zeros(
            logits.shape[0],
            dtype=torch.long,
            device=logits.device
        )

        loss = F.cross_entropy(logits, labels)

        acc_top1 = (
            logits.argmax(dim=1) == 0
        ).float().mean()

        self.log(
            f"{mode}_acc_top1",
            acc_top1,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            batch_size=im_q.shape[0]
        )

        if mode == "train":
            self._dequeue_and_enqueue(k)

        return loss
    def training_step(self, batch, batch_idx):
        loss = self.moco_loss(
            batch,
            mode="train"
        )

        return loss
    def validation_step(self, batch, batch_idx):
        loss = self.moco_loss(
            batch,
            mode="val"
        )

        return loss  
class LogisticRegression(pl.LightningModule):

    def __init__(self, feature_dim, num_classes, lr, weight_decay, max_epochs=100):
        super().__init__()
        self.save_hyperparameters()
        # Mapping from representation h to classes
        self.model = nn.Linear(feature_dim, num_classes)

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(),
                                lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        lr_scheduler = optim.lr_scheduler.MultiStepLR(optimizer,
                                                      milestones=[int(self.hparams.max_epochs*0.6),
                                                                  int(self.hparams.max_epochs*0.8)],
                                                      gamma=0.1)
        return [optimizer], [lr_scheduler]

    def _calculate_loss(self, batch, mode='train'):
        feats, labels = batch
        preds = self.model(feats)
        loss = F.cross_entropy(preds, labels)
        acc = (preds.argmax(dim=-1) == labels).float().mean()

        self.log(mode + '_acc', acc)
        return loss

    def training_step(self, batch, batch_idx):
        return self._calculate_loss(batch, mode='train')

    def validation_step(self, batch, batch_idx):
        self._calculate_loss(batch, mode='val')

    def test_step(self, batch, batch_idx):
        self._calculate_loss(batch, mode='test')

In [ ]:
#训练并返回训练好的模型
def train_moco(batch_size, max_epochs=50, **kwargs):
    checkpoint_callback = ModelCheckpoint(
        save_weights_only=True,
        mode="max",
        monitor="val_acc_top1"
    )

    trainer = pl.Trainer(
        default_root_dir=os.path.join(
            CHECKPOINT_PATH,
            "MoCo"
        ),
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        max_epochs=max_epochs,
        callbacks=[
            checkpoint_callback
        ]
    )

    train_loader = data.DataLoader(
        unlabeled_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
        num_workers=0
    )

    val_loader = data.DataLoader(
        train_data_contrast,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
        num_workers=0
    )

    model = MoCo(
        max_epochs=max_epochs,
        **kwargs
    )

    trainer.fit(model, train_loader, val_loader)

    model = MoCo.load_from_checkpoint(
        checkpoint_callback.best_model_path
    )

    return model
moco_model = train_moco(batch_size=32,
                            hidden_dim=128,
                            lr=5e-4,
                            temperature=0.07,
                            weight_decay=1e-4,
                            max_epochs=10)

In [ ]:
img_transforms = transforms.Compose([transforms.ToTensor(),
                                     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
train_img_data = STL10(root=DATASET_PATH, split='train', download=True,
                       transform=img_transforms)
test_img_data = STL10(root=DATASET_PATH, split='test', download=True,
                      transform=img_transforms)
#按类型准备好features(images)，labels
@torch.no_grad()
def prepare_data_features(model, dataset):
    # MoCo 下游任务使用 query encoder
    network = deepcopy(model.encoder_q)

    # 去掉 projection head，提取 ResNet backbone 的特征
    network.fc = nn.Identity()

    network.eval()
    network.to(device)

    data_loader = data.DataLoader(
        dataset,
        batch_size=64,
        num_workers=0,
        shuffle=False,
        drop_last=False
    )

    feats, labels = [], []

    for batch_imgs, batch_labels in tqdm(data_loader):
        batch_imgs = batch_imgs.to(device)

        batch_feats = network(batch_imgs)

        feats.append(batch_feats.cpu())
        labels.append(batch_labels)

    feats = torch.cat(feats, dim=0)
    labels = torch.cat(labels, dim=0)

    return data.TensorDataset(feats, labels)
train_feats_moco = prepare_data_features(moco_model,train_img_data)
test_feats_moco = prepare_data_features(moco_model,test_img_data)
def train_logreg(batch_size, train_feats_data, test_feats_data,  max_epochs=100, **kwargs):
    trainer = pl.Trainer(default_root_dir=os.path.join(CHECKPOINT_PATH, "LogisticRegression"),
                         accelerator="gpu" if str(device).startswith("cuda") else "cpu",
                         devices=1,
                         max_epochs=max_epochs,
                         precision="16-mixed",
                         callbacks=[ModelCheckpoint(save_weights_only=True, mode='max', monitor='val_acc'),
                                    LearningRateMonitor("epoch")],
                         enable_progress_bar=True,
                         check_val_every_n_epoch=10)
    trainer.logger._default_hp_metric = None

    # Data loaders
    train_loader = data.DataLoader(train_feats_data, batch_size=batch_size, shuffle=True,
                                   drop_last=False, pin_memory=True, num_workers=0)
    test_loader = data.DataLoader(test_feats_data, batch_size=batch_size, shuffle=False,
                                  drop_last=False, pin_memory=True, num_workers=0)

    model = LogisticRegression(**kwargs)
    trainer.fit(model, train_loader, test_loader)
    model = LogisticRegression.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)

    # Test best model on train and validation set
    test_result = trainer.test(model, test_loader, verbose=False)
    result = { "test": test_result[0]["test_acc"]}

    return model, result
logreg_model, result = train_logreg(
    batch_size=64,
    max_epochs=100,
    train_feats_data=train_feats_moco,
    test_feats_data=test_feats_moco,
    feature_dim=train_feats_moco.tensors[0].shape[1],
    num_classes=10,
    lr=1e-3,
    weight_decay=1e-3
)